In [1]:
pip install pandas numpy sqlalchemy pymysql faker

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 740.9 kB/s eta 0:00:03
   ---------- ----------------------------- 0.5/2.0 MB 740.9 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 643.8 kB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.0 MB 778.1 kB/s eta 0:00:02
   -------------------------- ------------- 1.3/2.0 MB 826.9 kB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 826.9 kB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 826.9 kB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 826.9 kB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 625.6 kB/s eta 0:00:01
   -------------------

<span style="color:red; font-weight:bold">solving an E-commerce Customer Retention & Profitability Analysis problem for a subscription-based retail company</span>


<span style="color:purple; font-weight:bold">Project Qiuz</span> Our customer acquisition costs (CAC) increased by 25% this year. We need to identify which customer segments have the highest Lifetime Value (LTV), detect geographic regions with declining profit margins, and predict which customers are at high risk of churning next month.

<span style="color:red; font-weight:bold">Project Roadmap</span>

<span style="color:purple; font-weight:bold">Step 1:</span> Write a Python script to generate a realistic, messy dataset and export it to MySQL.

<span style="color:purple; font-weight:bold">Step 2:</span> Use Python (scikit-learn) to calculate a Churn Risk Score for each customer.

<span style="color:purple; font-weight:bold">Step 3:</span> Write MySQL queries to build a relational star schema and calculate business metrics.

<span style="color:purple; font-weight:bold">Step 4:</span> Connect MySQL to Power BI and build advanced DAX measures.

<span style="color:purple; font-weight:bold">Step 5:</span> Design the final dashboard and write the GitHub/Upwork documentation.

<span style="color:red; font-weight:bold">Step 1: Generating the Data & MySQL Setup</span>

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker
from sqlalchemy import create_engine

# Initialize tool
fake = Faker()
np.random.seed(42)
num_customers = 1000

# 1. Create Customers DataFrame
customer_data = {
    'customer_id': range(1001, 1001 + num_customers),
    'signup_date': [fake.date_between(start_date='-2y', end_date='-6m') for _ in range(num_customers)],
    'country': np.random.choice(['United States', 'United Kingdom', 'Germany', 'Canada', 'Kenya'], size=num_customers, p=[0.4, 0.2, 0.15, 0.15, 0.10]),
    'age': np.random.randint(18, 70, size=num_customers),
    'account_type': np.random.choice(['Standard', 'Premium', 'VIP'], size=num_customers, p=[0.6, 0.3, 0.1])
}
df_customers = pd.DataFrame(customer_data)

# 2. Create Orders DataFrame (Simulating history)
order_records = []
order_id_counter = 50001

for _, row in df_customers.iterrows():
    # Give each customer a random number of orders based on their account type
    num_orders = np.random.randint(1, 15) if row['account_type'] == 'Standard' else np.random.randint(5, 30)
    
    for _ in range(num_orders):
        order_date = fake.date_between(start_date=row['signup_date'], end_date='today')
        sales_amount = round(np.random.uniform(15.0, 500.0), 2)
        # Introduce a few null profit margins for data cleaning practice later
        profit_margin = np.nan if np.random.rand() < 0.03 else round(sales_amount * np.random.uniform(0.1, 0.4), 2)
        
        order_records.append({
            'order_id': order_id_counter,
            'customer_id': row['customer_id'],
            'order_date': order_date,
            'sales_amount': sales_amount,
            'profit_amount': profit_margin,
            'delivery_status': np.random.choice(['Delivered', 'Shipped', 'Cancelled'], p=[0.90, 0.07, 0.03])
        })
        order_id_counter += 1

df_orders = pd.DataFrame(order_records)

print(f"Generated {len(df_customers)} customers and {len(df_orders)} order records successfully!")


Generated 1000 customers and 11237 order records successfully!


In [11]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# --- 1. ENTER YOUR MYSQL CONNECTION DETAILS HERE ---
USER = "root"          # Change to your MySQL username
PASSWORD = "Sijui254."  # Change to your MySQL password
HOST = "127.0.0.1"     # Localhost
PORT = "3306"          # Default MySQL port
DB_NAME = "ecommerce_portfolio"

# Create the SQLAlchemy engine connection string
connection_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# --- 2. CLEANING THE DATA (Demonstrating Data Quality Skills) ---
# Check for null values in profit_amount and fill them using an average 25% margin
default_margin = df_orders['sales_amount'] * 0.25
df_orders['profit_amount'] = df_orders['profit_amount'].fillna(default_margin).round(2)

# Ensure data types are optimized for database injection
df_customers['signup_date'] = pd.to_datetime(df_customers['signup_date'])
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'])

# --- 3. STEP 2: PREDICTIVE ML ENGINE (Predicting Churn Risk) ---
# Let's calculate total spending and days since last order for feature engineering
max_date = df_orders['order_date'].max()
customer_features = df_orders.groupby('customer_id').agg(
    recency=('order_date', lambda x: (max_date - x.max()).days),
    frequency=('order_id', 'count'),
    monetary=('sales_amount', 'sum')
).reset_index()

# Simple business logic heuristic acting as a placeholder ML predictive score
# High recency (days since last order) combined with lower frequency means high churn risk
customer_features['churn_risk_score'] = (
    (customer_features['recency'] * 0.6) + 
    ((1 / (customer_features['frequency'] + 1)) * 1000 * 0.4)
)
# Normalize score between 0.0 and 1.0
c_min, c_max = customer_features['churn_risk_score'].min(), customer_features['churn_risk_score'].max()
customer_features['churn_risk_score'] = ((customer_features['churn_risk_score'] - c_min) / (c_max - c_min)).round(4)

# Merge predictions back into the customer dataframe
df_customers_final = pd.merge(df_customers, customer_features[['customer_id', 'churn_risk_score']], on='customer_id', how='left')
df_customers_final['churn_risk_score'] = df_customers_final['churn_risk_score'].fillna(0.5) # baseline fallback

# --- 4. EXPORTING TABLES TO MYSQL ---
try:
    print("Uploading tables to MySQL database...")
    
    # Uploading dataframes into database tables
    df_customers_final.to_sql(name='dim_customers', con=engine, if_exists='replace', index=False)
    df_orders.to_sql(name='fact_orders', con=engine, if_exists='replace', index=False)
    
    print("✨ Success! 'dim_customers' and 'fact_orders' tables have been created and populated.")
except Exception as e:
    print(f"❌ Error uploading to MySQL: {e}")


Uploading tables to MySQL database...
✨ Success! 'dim_customers' and 'fact_orders' tables have been created and populated.


<span style="color:purple; font-weight:bold">The Business Analysis SQL Script</span>

USE ecommerce_portfolio;

-- ====================================================================
-- PORTFOLIO QUERY: CUSTOMER LIFETIME VALUE & GEOGRAPHIC PROFIT MARGINS
-- ====================================================================

WITH Customer_Metrics AS (
    -- Step 1: Aggregate order data per customer using CTE
    SELECT 
        customer_id,
        COUNT(order_id) AS total_orders,
        SUM(sales_amount) AS total_spent,
        SUM(profit_amount) AS total_profit,
        MAX(order_date) AS last_purchase_date
    FROM fact_orders
    WHERE delivery_status != 'Cancelled' -- Exclude cancelled orders from financial KPIs
    GROUP BY customer_id
),
Regional_Analysis AS (
    -- Step 2: Combine aggregates with customer attributes and calculate margins
    SELECT 
        c.customer_id,
        c.country,
        c.account_type,
        c.churn_risk_score,
        cm.total_orders,
        cm.total_spent,
        cm.total_profit,
        ROUND((cm.total_profit / cm.total_spent) * 100, 2) AS profit_margin_pct
    FROM dim_customers c
    JOIN Customer_Metrics cm ON c.customer_id = cm.customer_id
)
-- Step 3: Use Window Functions to rank customers by profitability within their country
SELECT 
    country,
    customer_id,
    account_type,
    total_spent AS lifetime_value_ltv,
    profit_margin_pct,
    churn_risk_score,
    DENSE_RANK() OVER (PARTITION BY country ORDER BY total_spent DESC) AS customer_rank_in_country
FROM Regional_Analysis
ORDER BY country ASC, lifetime_value_ltv DESC;


<span style="color:purple; font-weight:bold">Step 4: Connecting to Power BI</span>